In [12]:
!pip install kagglehub

In [13]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("wordsforthewise/lending-club")

print("Path to dataset files:", path)

100%|██████████| 1.26G/1.26G [01:58<00:00, 11.5MB/s] 

Extracting files...


Path to dataset files: C:\Users\ganes\.cache\kagglehub\datasets\wordsforthewise\lending-club\versions\3


In [14]:
# Unzipping the Data After Downloading

In [18]:
# Move or extract files from the kagglehub download path to data/
from pathlib import Path
import shutil, zipfile

# `path` variable is set by the kagglehub downloader cell above; fall back to a known cache location if needed
src = Path(globals().get('path', ''))
if not src or not src.exists():
    possible = Path.home() / '.cache' / 'kagglehub' / 'datasets' / 'wordsforthewise' / 'lending-club' / 'versions'
    # pick the newest version directory
    if possible.exists():
        versions = sorted([d for d in possible.iterdir() if d.is_dir()], reverse=True)
        if versions:
            src = versions[0]

if not src or not src.exists():
    print('Could not locate the downloaded dataset path. Current `path` variable:', globals().get('path'))
else:
    data_dir = Path('data')
    data_dir.mkdir(parents=True, exist_ok=True)
    print(f'Copying files from {src} to {data_dir}...')
    for item in src.iterdir():
        if item.suffix == '.zip':
            print(f'Found zip: {item} — extracting to {data_dir}')
            with zipfile.ZipFile(item, 'r') as z:
                z.extractall(data_dir)
        elif item.is_file():
            dest = data_dir / item.name
            shutil.copy2(item, dest)
            print(f'Copied {item.name} -> {dest}')
    print('Done. Files in data/:')
    for f in sorted(data_dir.iterdir()):
        print(' -', f.name)


Copying files from C:\Users\ganes\.cache\kagglehub\datasets\wordsforthewise\lending-club\versions\3 to data...
Copied accepted_2007_to_2018Q4.csv.gz -> data\accepted_2007_to_2018Q4.csv.gz
Copied rejected_2007_to_2018Q4.csv.gz -> data\rejected_2007_to_2018Q4.csv.gz
Done. Files in data/:
 - accepted_2007_to_2018Q4.csv.gz
 - rejected_2007_to_2018Q4.csv.gz


In [19]:
import pandas as pd
import numpy as np

In [26]:
df = pd.read_csv('accepted_2007_to_2018Q4.csv', low_memory=False, parse_dates=['issue_d'])
print(f"Data loaded successfully. Shape: {df.shape}")

C:\Users\ganes\AppData\Local\Temp\ipykernel_29956\1775497950.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df = pd.read_csv('accepted_2007_to_2018Q4.csv', low_memory=False, parse_dates=['issue_d'])


Data loaded successfully. Shape: (2260701, 151)


In [27]:
# --- 4. 🎯 Create the Target Variable ---
# This is the most important step for our helper model.
# The 'loan_status' column has many values (e.g., "Current", "In Grace Period").
# We only care about loans that are *finished*.
print("\nOriginal 'loan_status' values:")
print(df['loan_status'].value_counts())


Original 'loan_status' values:
loan_status
Fully Paid                                             1076751
Current                                                 878317
Charged Off                                             268559
Late (31-120 days)                                       21467
In Grace Period                                           8436
Late (16-30 days)                                         4349
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     40
Name: count, dtype: int64


In [28]:
# Filter for completed loans
completed_loans_df = df[df['loan_status'].isin(['Fully Paid', 'Charged Off'])].copy()

# Create the binary target variable
# 'Charged Off' is a default (1), 'Fully Paid' is not (0).
completed_loans_df['is_default'] = (completed_loans_df['loan_status'] == 'Charged Off').astype(int)

print(f"\nFiltered data for completed loans. New shape: {completed_loans_df.shape}")
print("\nNew 'is_default' target variable counts:")
print(completed_loans_df['is_default'].value_counts())


Filtered data for completed loans. New shape: (1345310, 152)

New 'is_default' target variable counts:
is_default
0    1076751
1     268559
Name: count, dtype: int64


In [29]:
!pip install xgboost

from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import joblib
import pandas as pd 
import numpy as np

   ---------------------------------------- 0.0/72.0 MB ? eta -:--:--
   ---------------------------------------- 0.8/72.0 MB 5.7 MB/s eta 0:00:13
   - -------------------------------------- 2.1/72.0 MB 6.1 MB/s eta 0:00:12
   - -------------------------------------- 3.4/72.0 MB 6.1 MB/s eta 0:00:12
   -- ------------------------------------- 4.2/72.0 MB 5.5 MB/s eta 0:00:13
   --- ------------------------------------ 5.5/72.0 MB 5.5 MB/s eta 0:00:13
   --- ------------------------------------ 7.1/72.0 MB 5.9 MB/s eta 0:00:11
   ---- ----------------------------------- 8.9/72.0 MB 6.4 MB/s eta 0:00:10
   ----- ---------------------------------- 10.7/72.0 MB 6.6 MB/s eta 0:00:10
   ------- -------------------------------- 12.8/72.0 MB 7.0 MB/s eta 0:00:09
   -------- ------------------------------- 14.9/72.0 MB 7.3 MB/s eta 0:00:08
   --------- ------------------------------ 17.3/72.0 MB 7.6 MB/s eta 0:00:08
   ---------- ----------------------------- 19.7/72.0 MB 7.9 MB/s eta 0:00:07
 

In [30]:
features = [
    'loan_amnt',
    'int_rate',
    'annual_inc',
    'dti',
    'fico_range_low',
    'term'
]
target = 'is_default'

In [31]:
completed_loans_df['term'] = completed_loans_df['term'].str.strip().str.replace(' months', '').astype(float)
model_df = completed_loans_df[features + [target]].copy()

In [32]:
original_rows = model_df.shape[0]
model_df = model_df.dropna()
print(f"\nDropped {original_rows - model_df.shape[0]} rows with missing values.")
print(f"Final modeling data shape: {model_df.shape}")


Dropped 374 rows with missing values.
Final modeling data shape: (1344936, 7)


In [33]:
X = model_df[features]
y = model_df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [34]:
print(f"\nTraining set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")


Training set size: 1075948
Test set size: 268988


In [ ]:
print("\nTraining the XGBoost Default Model...")




Training the XGBoost Default Model...


In [39]:
default_model = XGBClassifier(
    n_estimators=4000,      # Can use more trees w/ early stopping
    learning_rate=0.1,
    max_depth=50,           # Can often go a bit deeper than standard GB
    random_state=42,
    tree_method='hist',    # Use 'hist' for CPU speed. Use 'gpu_hist' if GPU is enabled.
    early_stopping_rounds=10 # Stop if validation score doesn't improve
)


In [40]:
# Let's train on a 200k sample to make it fast
X_train_sample = X_train.sample(n=min(200000, X_train.shape[0]), random_state=42)
y_train_sample = y_train[X_train_sample.index]

# We need a small validation set for early stopping
X_val_sample = X_test.sample(n=min(50000, X_test.shape[0]), random_state=42)
y_val_sample = y_test[X_val_sample.index]

In [41]:
default_model.fit(
    X_train_sample, 
    y_train_sample, 
    eval_set=[(X_val_sample, y_val_sample)], 
    verbose=False
)
print("Model training complete.")

Model training complete.


In [42]:
# 6. Evaluate Model
y_pred = default_model.predict(X_test)
print("\n--- Default Model Evaluation (XGBoost) ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred, target_names=['Fully Paid (0)', 'Defaulted (1)']))

# 7. Save the Model
model_filename = 'default_model.joblib'
joblib.dump(default_model, model_filename)
print(f"\nModel saved as '{model_filename}'")


--- Default Model Evaluation (XGBoost) ---
Accuracy: 0.8000
                precision    recall  f1-score   support

Fully Paid (0)       0.80      1.00      0.89    215290
 Defaulted (1)       0.47      0.01      0.02     53698

      accuracy                           0.80    268988
     macro avg       0.63      0.50      0.46    268988
  weighted avg       0.73      0.80      0.72    268988


Model saved as 'default_model.joblib'
